In [1]:
import numpy as np
import keras
import string
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction
import random

## data can be downloaded from http://www.manythings.org/anki/pol-eng.zip

In [2]:
batch_size = 64  # Batch size for training.
epochs = 100  # Number of epochs to train for.
latent_dim = 256  # Latent dimensionality of the encoding space.
num_samples = 10000  # Number of samples to train on.
# Path to the data txt file on disk.
data_path = "../pol-eng/pol.txt"

In [3]:
# Vectorize the data.
input_texts = []
target_texts = []
input_characters = set()
target_characters = set()
with open(data_path, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[: min(num_samples, len(lines) - 1)]:
    input_text, target_text, _ = line.split("\t")
    # We use "tab" as the "start sequence" character
    # for the targets, and "\n" as "end sequence" character.
    target_text = "\t" + target_text + "\n"
    input_texts.append(input_text)
    target_texts.append(target_text)
    for char in input_text:
        if char not in input_characters:
            input_characters.add(char)
    for char in target_text:
        if char not in target_characters:
            target_characters.add(char)

input_characters = sorted(list(input_characters))
target_characters = sorted(list(target_characters))
num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Number of samples:", len(input_texts))
print("Number of unique input tokens:", num_encoder_tokens)
print("Number of unique output tokens:", num_decoder_tokens)
print("Max sequence length for inputs:", max_encoder_seq_length)
print("Max sequence length for outputs:", max_decoder_seq_length)

input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens),
    dtype="float32",
)
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens),
    dtype="float32",
)

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    encoder_input_data[i, t + 1 :, input_token_index[" "]] = 1.0
    for t, char in enumerate(target_text):
        # decoder_target_data is ahead of decoder_input_data by one timestep
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            # decoder_target_data will be ahead by one timestep
            # and will not include the start character.
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0
    decoder_input_data[i, t + 1 :, target_token_index[" "]] = 1.0
    decoder_target_data[i, t:, target_token_index[" "]] = 1.0

Number of samples: 10000
Number of unique input tokens: 70
Number of unique output tokens: 87
Max sequence length for inputs: 19
Max sequence length for outputs: 52


## Build the Model

In [4]:
# Define an input sequence and process it.
encoder_inputs = keras.Input(shape=(None, num_encoder_tokens))
encoder = keras.layers.LSTM(latent_dim, return_state=True)
encoder_outputs, state_h, state_c = encoder(encoder_inputs)

# We discard `encoder_outputs` and only keep the states.
encoder_states = [state_h, state_c]

# Set up the decoder, using `encoder_states` as initial state.
decoder_inputs = keras.Input(shape=(None, num_decoder_tokens))

# We set up our decoder to return full output sequences,
# and to return internal states as well. We don't use the
# return states in the training model, but we will use them in inference.
decoder_lstm = keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = keras.layers.Dense(num_decoder_tokens, activation="softmax")
decoder_outputs = decoder_dense(decoder_outputs)

# Define the model that will turn
# `encoder_input_data` & `decoder_input_data` into `decoder_target_data`
model = keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

In [5]:
model.compile(
    optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"]
)
model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2,
)
# Save model
model.save("s2s_model.keras")

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 14s 111ms/step - accuracy: 0.6739 - loss: 1.6955 - val_accuracy: 0.6562 - val_loss: 1.6051
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 14s 116ms/step - accuracy: 0.7155 - loss: 1.1153 - val_accuracy: 0.6679 - val_loss: 1.2205
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 15s 119ms/step - accuracy: 0.7385 - loss: 0.9865 - val_accuracy: 0.6936 - val_loss: 1.1745
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 128ms/step - accuracy: 0.7664 - loss: 0.8761 - val_accuracy: 0.7245 - val_loss: 1.0037
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 127ms/step - accuracy: 0.7849 - loss: 0.7846 - val_accuracy: 0.7414 - val_loss: 0.9259
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 126ms/step - accuracy: 0.7922 - loss: 0.7495 - val_accuracy: 0.7456 - val_loss: 0.9035
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 16s 131ms/step - accuracy: 0.8000 - loss: 0.7071 - val_accuracy: 0.7515 - val_loss: 0.8833
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 18s 147ms/step - accuracy: 0.8054 -

In [6]:
# Define sampling models
# Restore the model and construct the encoder and decoder.
model = keras.models.load_model("s2s_model.keras")

encoder_inputs = model.input[0]  # input_1
encoder_outputs, state_h_enc, state_c_enc = model.layers[2].output  # lstm_1
encoder_states = [state_h_enc, state_c_enc]
encoder_model = keras.Model(encoder_inputs, encoder_states)

decoder_inputs = model.input[1]  # input_2
decoder_state_input_h = keras.Input(shape=(latent_dim,))
decoder_state_input_c = keras.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_lstm = model.layers[3]
decoder_outputs, state_h_dec, state_c_dec = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs
)
decoder_states = [state_h_dec, state_c_dec]
decoder_dense = model.layers[4]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = keras.Model(
    [decoder_inputs] + decoder_states_inputs, [decoder_outputs] + decoder_states
)

# Reverse-lookup token index to decode sequences back to
# something readable.
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())


def decode_sequence(input_seq):
    # Encode the input as state vectors.
    states_value = encoder_model.predict(input_seq, verbose=0)

    # Generate empty target sequence of length 1.
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    # Populate the first character of target sequence with the start character.
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_condition = False
    decoded_sentence = ""
    while not stop_condition:
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, verbose=0
        )

        # Sample a token
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length:
            stop_condition = True

        # Update the target sequence (of length 1).
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0

        # Update states
        states_value = [h, c]
    return decoded_sentence

In [7]:
for seq_index in range(20):
    # Take one sequence (part of the training set)
    # for trying out decoding.
    input_seq = encoder_input_data[seq_index : seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("-")
    print("Input sentence:", input_texts[seq_index])
    print("Decoded sentence:", decoded_sentence)

-
Input sentence: Go.
Decoded sentence: Uki.

-
Input sentence: Hi.
Decoded sentence: Cześć.

-
Input sentence: Run!
Decoded sentence: Uciekaj!

-
Input sentence: Run.
Decoded sentence: Uciekaj.

-
Input sentence: Run.
Decoded sentence: Uciekaj.

-
Input sentence: Who?
Decoded sentence: Kto znienił?

-
Input sentence: Wow!
Decoded sentence: Ogedz!

-
Input sentence: Wow!
Decoded sentence: Ogedz!

-
Input sentence: Duck!
Decoded sentence: Oduż się!

-
Input sentence: Fire!
Decoded sentence: Strzelaj!

-
Input sentence: Fire!
Decoded sentence: Strzelaj!

-
Input sentence: Fire!
Decoded sentence: Strzelaj!

-
Input sentence: Help!
Decoded sentence: Pomocy!

-
Input sentence: Hide.
Decoded sentence: Przekrzytaj się.

-
Input sentence: Jump!
Decoded sentence: Skacz!

-
Input sentence: Jump.
Decoded sentence: Widziej.

-
Input sentence: Stay.
Decoded sentence: Zapisz się.

-
Input sentence: Stop!
Decoded sentence: Stań z brawanenie sary!

-
Input sentence: Stop!
Decoded sentence: Stań z braw

# BLEU Evaluation

In [17]:
test_size = 100
test_input_texts = []
test_target_texts = []
with open(data_path, "r", encoding="utf-8") as f:
    lines = f.read().split("\n")
for line in lines[num_samples:num_samples + test_size]:
    input_text, target_text, _ = line.split("\t")
    target_text = "\t" + target_text + "\n"
    test_input_texts.append(input_text)
    test_target_texts.append(target_text)

test_encoder_input_data = np.zeros(
    (len(test_input_texts), test_size, num_encoder_tokens),
    dtype="float32",
)

for i, (input_text, target_text) in enumerate(zip(test_input_texts, test_target_texts)):
    for t, char in enumerate(input_text):
        try:
            test_encoder_input_data[i, t, input_token_index[char]] = 1.0
        except KeyError:
            print(i)
    test_encoder_input_data[i, t + 1 :, input_token_index[" "]] = 1.0

In [ ]:
def get_tokens(sentence):
    tokens = sentence.strip().split()
    return [token for token in tokens if token not in string.punctuation]

In [95]:
pred_tokens = []
ref_tokens = []

rand_elems = random.sample(range(len(test_input_texts)), 50)

for i in rand_elems:
    input_seq = test_encoder_input_data[i : i + 1]
    decoded_sentence = decode_sequence(input_seq)
    pred_tokens.append(get_tokens(decoded_sentence))

    input_text = test_input_texts[i]
    target_text = test_target_texts[i]

    ref_tokens.append([get_tokens(target_text)])

In [97]:
eval_on_top_n = 100

pred_tokens = []
ref_tokens = []

rand_elems = random.sample(range(len(input_texts)), eval_on_top_n)

for i in rand_elems:
    input_seq = encoder_input_data[i : i + 1]
    decoded_sentence = decode_sequence(input_seq)
    pred_tokens.append(get_tokens(decoded_sentence))

    input_text = input_texts[i]
    target_text = target_texts[i]

    print(f"{i} Input sentence: {input_text} ==> {target_text} VS {decoded_sentence}")

    ref_tokens.append([get_tokens(target_text)])

5429 Input sentence: I like climbing. ==> 	Lubię się wspinać.
 VS Lubię się bezpieszcia.

2756 Input sentence: She is happy. ==> 	Jest szczęśliwa.
 VS Ona jest młoda.

7397 Input sentence: They're not sure. ==> 	Nie są pewne.
 VS Nie są pewni.

4366 Input sentence: I'm a bachelor. ==> 	Jestem kawalerem.
 VS Jestem powiedziały na nas.

7596 Input sentence: We both love Tom. ==> 	Obie kochamy Toma.
 VS Nie przemyliśmy się.

8067 Input sentence: He knows too much. ==> 	Wie za dużo.
 VS Podłunij biogolę.

7162 Input sentence: My car is German. ==> 	Mój samochód jest niemiecki.
 VS Moje dziecko jest chore.

204 Input sentence: I slept. ==> 	Spałam.
 VS Spałam.

9218 Input sentence: We're from Boston. ==> 	Jesteśmy z Bostonu.
 VS Jesteśmy utwarty.

860 Input sentence: Pipe down! ==> 	Zamknąć japy!
 VS Przyszej wieczni!

8136 Input sentence: How's your mother? ==> 	Jak się miewa Twoja mama?
 VS Jak się miewa pies o zarmą.

3913 Input sentence: You came back. ==> 	Wróciłeś.
 VS Masz czas.

308

In [98]:
smoothie = SmoothingFunction().method4
bleu = corpus_bleu(ref_tokens, pred_tokens, smoothing_function=smoothie)
print(f"BLEU: {bleu}")

BLEU: 0.06709453114343732
